# BR04C — Preserve exact Shaoxing PULSE images for all generators

Use once after BR02A/PULSE generation and before running the other generators in BR04B. The current loader redraws PNGs in each environment, which can change the PNG hash and stop candidate provenance verification. This notebook verifies and archives exact original images. It does not run a model, modify candidate hashes, change packets, or relax leakage checks.

**First run cells 1–3 in the exact PULSE environment used by BR02A.** If images were overwritten by a modern kernel, the archive step reconstructs into temporary files and accepts only byte-identical PNGs. It leaves the shared cohort image directory untouched. If exact reconstruction fails, restore the original rendering environment; do not loosen the check. Archive preparation is resumable. No GPU is required.


In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None: raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup
CONTEXT_CONFIG_PATH = Path('/path/to/repository/outputs/real_run_001/experiment02_v1/context_fit/v1/shaoxing/config.json')
CFG = load_config(CONTEXT_CONFIG_PATH)
parent_setup(CFG)
assert CFG['cohort'] == 'shaoxing'
from extensions.query_image_archive import diagnose, prepare_archive
print('Kernel:', sys.executable)
print('Child run:', CFG['run_dir'])


In [ ]:
# Read-only diagnosis: never regenerates or overwrites images.
for result in diagnose(CFG, probes=3):
    print(result)
# A false source_tensor_matches means a waveform/preprocessing issue: stop.
# True source tensor plus false current_png_matches is a rendering/file mismatch.


In [ ]:
# Run this in the original PULSE environment. No model/checkpoint loading needed.
RECONSTRUCT_IN_ORIGINAL_PULSE_ENVIRONMENT = True
result = prepare_archive(CFG, reconstruct=RECONSTRUCT_IN_ORIGINAL_PULSE_ENVIRONMENT,
                         max_run_hours=2.0)
print(result)
if not result['complete']:
    raise RuntimeError('Archive incomplete; rerun this cell before proceeding.')
print('READY: return to BR04B in the desired generator environment.')


## Return to BR04B

Keep MODEL_KEY set to the desired generator and CONTEXT_CONFIG_PATH set to the same Shaoxing child config above. Restart/reload the notebook in that model's environment and run its setup/model-loading cells. Replace only the generation cell with the code below. Use this wrapper for llama_ecg, llama_base and qwen38 so all receive the original PULSE image bytes. Keep ENABLE_THINKING=False. Inspect five reports, then set MAX_NEW_REPORTS=0. Existing successful records are skipped only when their provenance matches.

Do not edit queries.py, context_fit.py, frozen protocol files or historical report hashes. Do not rerun old code that writes shared ECG images concurrently. Existing completed PULSE outputs remain valid because the archived inputs are verified against their candidate-extraction image identities. The new wrapper logs its own archive provenance beside the reports.


In [ ]:
# Copy this cell into BR04B after FIT_CFG and runtime have been created.
# It intentionally requires the current BR04B model runtime.
from extensions.query_image_archive import generate_archived_context_fit
from generation import folder_for
assert FIT_CFG['cohort'] == 'shaoxing'
MAX_NEW_REPORTS = 5  # After pilot succeeds, set 0 for the remaining reports.
RETRY_ERRORS = False
MAX_RUN_HOURS = 30.0  # Must be below the allocation time still remaining.
MAX_NEW_TOKENS = 1024 if MODEL_KEY == 'qwen38' else 384
print(generate_archived_context_fit(FIT_CFG, runtime, max_new=MAX_NEW_REPORTS,
    retry_errors=RETRY_ERRORS, max_run_hours=MAX_RUN_HOURS, max_new_tokens=MAX_NEW_TOKENS))
FOLDER = folder_for(FIT_CFG, MODEL_KEY, ENABLE_THINKING, MAX_NEW_TOKENS)
print('Reports:', FOLDER / 'reports.jsonl')
